# Step 2: Hybrid ResNet50 + CBAM Training for 5-Stage DR Grading (Google Colab T4 GPU)

Dedicated Colab-ready training & evaluation notebook for **Hybrid ResNet50 + CBAM (Channel & Spatial Attention)** on APTOS 2019:
- **Runtime:** Make sure to select `Runtime -> Change runtime type -> T4 GPU` in Google Colab.
- **Auto-Save:** Best Validation QWK checkpoint (`best_dr_model_{key}.pth`) is automatically saved to **Google Drive** (`/content/drive/MyDrive/RetinaRAG_DR_Project/models/`).
- **Metrics:** Quadratic Weighted Kappa (QWK), Accuracy, Macro-F1, Referable DR Recall (`Grade >= 2`), & Confusion Matrix.


### 1. Google Colab Setup, GPU Check & Auto-Clone Repo


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://gitlab.com/Shariar-Ahamed/hybrid-retinal-dr-detection-xai.git'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/RetinaRAG_DR_Project')
    (DRIVE_PROJECT_DIR / 'models').mkdir(parents=True, exist_ok=True)
    (DRIVE_PROJECT_DIR / 'data').mkdir(parents=True, exist_ok=True)

    ROOT_DIR = Path('/content/hybrid-retinal-dr-detection-xai')
    if not ROOT_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(ROOT_DIR), 'pull'], check=True)
    os.chdir(ROOT_DIR)
    print('[Colab] Repo ready at:', ROOT_DIR)
    print('[Colab] Google Drive backup folder:', DRIVE_PROJECT_DIR)
else:
    ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DRIVE_PROJECT_DIR = None
    print('[Local] Project Root:', ROOT_DIR)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)
if torch.cuda.is_available():
    print('GPU Name:', torch.cuda.get_device_name(0))


### 2. Download APTOS 2019 & Ensure `metadata.csv` Exists


In [ ]:
# Download & unzip Kaggle APTOS 2019 directly to fast Colab SSD (/content/aptos2019)
import shutil

APTOS_DIR = Path('/content/aptos2019') if IN_COLAB else (ROOT_DIR / 'data' / 'aptos2019')

if IN_COLAB and not (APTOS_DIR / 'train.csv').exists():
    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(exist_ok=True)
    drive_kaggle = Path('/content/drive/MyDrive/kaggle.json')
    if drive_kaggle.exists():
        shutil.copy2(drive_kaggle, kaggle_dir / 'kaggle.json')
        print('[+] Loaded kaggle.json from Google Drive!')
    elif not (kaggle_dir / 'kaggle.json').exists():
        from google.colab import files
        print('[i] Please upload your kaggle.json API token:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            shutil.move(fname, kaggle_dir / 'kaggle.json')
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)

    APTOS_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'competitions', 'download', '-c', 'aptos2019-blindness-detection', '-p', str(APTOS_DIR)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(APTOS_DIR / 'aptos2019-blindness-detection.zip'), '-d', str(APTOS_DIR)], check=True)
    print('[+] APTOS 2019 extracted to:', APTOS_DIR)
else:
    print('[+] APTOS 2019 directory ready at:', APTOS_DIR)

from src.preprocess import generate_aptos2019_metadata, DR_CLASS_NAMES
METADATA_PATH = ROOT_DIR / 'data' / 'metadata.csv'
if not METADATA_PATH.exists():
    if IN_COLAB and DRIVE_PROJECT_DIR is not None and (DRIVE_PROJECT_DIR / 'data' / 'metadata.csv').exists():
        shutil.copy2(DRIVE_PROJECT_DIR / 'data' / 'metadata.csv', METADATA_PATH)
        print('[+] Restored metadata.csv from Google Drive!')
    else:
        generate_aptos2019_metadata(aptos_dir=str(APTOS_DIR), val_size=0.10, test_size=0.10, seed=42)
        if IN_COLAB and DRIVE_PROJECT_DIR is not None:
            shutil.copy2(METADATA_PATH, DRIVE_PROJECT_DIR / 'data' / 'metadata.csv')


### 3. Experiment Configuration & DataLoaders


In [ ]:
import time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.hybrid_model import HybridDRPredictor
from src.train import RetinalFundusDataset, train_epoch, evaluate, set_seed

set_seed(42)

BACKBONE = 'resnet50'
USE_BEN_GRAHAM = True
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4

MODELS_DIR = ROOT_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)
CKPT_NAME = f'best_dr_model_{BACKBONE}.pth'
SAVE_PATH = MODELS_DIR / CKPT_NAME
DRIVE_SAVE_PATH = (DRIVE_PROJECT_DIR / 'models' / CKPT_NAME) if (IN_COLAB and DRIVE_PROJECT_DIR is not None) else None

df = pd.read_csv(METADATA_PATH)
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

class_counts = np.maximum(np.array([(train_df['label'] == c).sum() for c in range(5)], dtype=np.float32), 1.0)
class_weights = torch.tensor(class_counts.sum() / (5.0 * class_counts), dtype=torch.float32).to(device)
print('Dataset Split -> Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))
print('Class Weights (Grades 0-4):', class_weights.cpu().numpy().round(3))

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(RetinalFundusDataset(train_df, APTOS_DIR, train_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(RetinalFundusDataset(val_df, APTOS_DIR, eval_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(RetinalFundusDataset(test_df, APTOS_DIR, eval_tf, USE_BEN_GRAHAM, IMG_SIZE), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


### 4. Train Hybrid CNN-CBAM Model & Auto-Save Best QWK Checkpoint to Google Drive


In [ ]:
model = HybridDRPredictor(backbone=BACKBONE, num_classes=5, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_qwk, patience_cnt = -1.0, 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_qwk': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_qwk, val_f1 = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_qwk)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_qwk'].append(val_qwk)

    print(f'Epoch {epoch:02d}/{EPOCHS} [{time.time()-t0:.1f}s] | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val QWK: {val_qwk:.4f} | F1: {val_f1:.4f}')
    if val_qwk > best_qwk:
        best_qwk = val_qwk
        patience_cnt = 0
        torch.save(model.state_dict(), SAVE_PATH)
        if DRIVE_SAVE_PATH is not None:
            shutil.copy2(SAVE_PATH, DRIVE_SAVE_PATH)
            print(f'  -> Saved best checkpoint (QWK={best_qwk:.4f}) to local & Google Drive!')
        else:
            print(f'  -> Saved best checkpoint (QWK={best_qwk:.4f}) to {SAVE_PATH}')
    else:
        patience_cnt += 1
        if patience_cnt >= 5:
            print('Early stopping triggered.')
            break


### 5. Training Curves, Hold-Out Test Evaluation & Confusion Matrix


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_title(f'{BACKBONE} (Hybrid CBAM) — Loss'); ax1.legend()
ax2.plot(history['val_qwk'], label='Val QWK', color='green')
ax2.plot(history['val_acc'], label='Val Accuracy', color='blue')
ax2.set_title(f'{BACKBONE} (Hybrid CBAM) — QWK & Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.eval()
preds, targets = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        out = model(imgs.to(device))
        preds.extend(torch.argmax(out, dim=1).cpu().numpy().tolist())
        targets.extend(lbls.numpy().tolist())

preds_np, targets_np = np.array(preds), np.array(targets)
test_qwk = cohen_kappa_score(targets_np, preds_np, weights='quadratic')
ref_true = (targets_np >= 2).astype(int)
ref_pred = (preds_np >= 2).astype(int)
ref_sens = float(np.sum((ref_true == 1) & (ref_pred == 1)) / max(np.sum(ref_true == 1), 1))

print(f'Hold-Out Test Accuracy: {np.mean(preds_np == targets_np)*100:.2f}%')
print(f'Hold-Out Test QWK:      {test_qwk:.4f}')
print(f'Referable DR Recall:    {ref_sens*100:.2f}%')
print('\n' + classification_report(targets_np, preds_np, target_names=[DR_CLASS_NAMES[i] for i in range(5)]))

cm = confusion_matrix(targets_np, preds_np)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[DR_CLASS_NAMES[i] for i in range(5)], yticklabels=[DR_CLASS_NAMES[i] for i in range(5)])
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title(f'{BACKBONE} (Hybrid CBAM) — Hold-Out Confusion Matrix')
plt.tight_layout(); plt.show()


### 6. Optional: Download Trained `.pth` Checkpoint Directly to Your PC


In [ ]:
if IN_COLAB:
    from google.colab import files
    print('Also saved permanently in Google Drive at:', DRIVE_SAVE_PATH)
    # Uncomment the line below to download directly to your browser:
    # files.download(str(SAVE_PATH))
